# ART calculations for `Yersinia Pestis` bacteria

In [34]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [35]:
import sys

# Completely suppress stderr output
sys.stderr = open(os.devnull, 'w')

# Now import everything
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
sys.path.append('.')   # Add local directory to access some of the functions
sys.path.append('/users/sghosh6/DTRA_project/MACAW/AutomatedRecommendationTool') # Make sure this is the location for the ART library

import warning_utils
warning_utils.filter_end_user_warnings()

In [37]:
import pandas as pd
import numpy as np

from rdkit import Chem
from art.core import RecommendationEngine
import art.utility as utils
import pickle
import cloudpickle
import matplotlib.pyplot as plt

### Define directories

In [49]:
#dataDir = '/code/DTRA_ART/DrugDesignData/'
dataDir = '/users/sghosh6/DTRA_project/MACAW/DTRA_ART/DrugDesignData/'
modelBuildingDataDir = os.path.join(dataDir, 'modelBuildingData/')
resultsDir = os.path.join(dataDir, 'Results/')
artResultDir = '/users/sghosh6/DTRA_project/MACAW/DTRA_ART/test_CLI/ART_results/model_outputs/'
os.makedirs(artResultDir, exist_ok=True)
#saveDir = os.path.join(resultsDir, "Yersinia_pestis/")
saveDir = os.path.join(resultsDir, "Yersinia_pestis_test/")
os.makedirs(saveDir, exist_ok=True)

### Extract the data for `Yersinia Pestis` into a data frame with `duplicate` SMILES

In [39]:
YersiniaPestisData_chEMBL_wMACAW = pd.read_csv(modelBuildingDataDir + "YersiniaPestisData_chEMBL_wMACAW.csv")
YersiniaPestisData_chEMBL_wMACAW 

,ID,compound_id,Smiles,pPotency,BacteriaClassifier,MACAW_1,MACAW_2,MACAW_3,MACAW_4,MACAW_5,...,MACAW_21,MACAW_22,MACAW_23,MACAW_24,MACAW_25,MACAW_26,MACAW_27,MACAW_28,MACAW_29,MACAW_30
0,1,CHEMBL2368935,CC(=O)NCCCC(=O)N[C@@H](Cc1ccccc1)C(=O)N1Cc2ccc...,3.707744,Yersinia_pestis,-0.481541,-0.191413,0.263104,-0.136817,0.102238,...,0.039121,-0.041539,-0.031224,0.008813,-0.011636,0.033561,-0.032778,-0.004993,-0.042450,0.004318
1,2,CHEMBL1255638,CC(=O)NCC(=O)N[C@@H](Cc1ccccc1)C(=O)N1Cc2ccccc...,3.657577,Yersinia_pestis,-0.482748,-0.192809,0.264683,-0.137308,0.104631,...,0.043566,-0.041686,-0.033906,0.011488,-0.013285,0.030705,-0.031670,-0.004712,-0.043531,-0.000071
2,3,CHEMBL1255640,CC(=O)N[C@@H](CCCCN)C(=O)N[C@@H](CCCCN)C(=O)N[...,4.882729,Yersinia_pestis,-0.482518,-0.193130,0.264661,-0.136940,0.103483,...,0.041069,-0.041919,-0.032173,0.008391,-0.013187,0.032686,-0.032475,-0.004502,-0.038330,0.001017
3,4,CHEMBL1255643,CC(C)C[C@H](NC(=O)[C@@H](N)CCCCN)C(=O)N1Cc2ccc...,3.823909,Yersinia_pestis,-0.481617,-0.189056,0.263961,-0.138981,0.106303,...,0.045950,-0.048610,-0.034550,0.008308,-0.008837,0.030580,-0.032571,-0.007995,-0.045730,-0.000904
4,5,CHEMBL1200699,C[C@H]1c2cccc(O)c2C(=O)C2=C(O)[C@]3(O)C(=O)C(C...,6.251812,Yersinia_pestis,-0.003173,0.051418,-0.072599,0.013071,-0.139534,...,0.128165,-0.502080,-0.145858,0.166413,-0.041240,0.018207,-0.031798,0.039600,-0.085370,-0.081963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,182,CHEMBL964,CCN(CC)C(=S)SSC(=S)N(CC)CC,4.494850,Yersinia_pestis,-0.082783,0.372704,0.080347,0.132415,0.060888,...,-0.003241,0.023322,0.023084,-0.031481,-0.070509,-0.061901,0.048196,-0.025893,-0.064942,-0.002196
149,183,CHEMBL4227290,CCCCCCCSSC(=S)N(CC)CC,4.494850,Yersinia_pestis,-0.086559,0.448239,0.090822,0.223673,0.081537,...,0.007137,-0.022239,0.004531,0.012768,0.060608,-0.008669,-0.121745,-0.060900,-0.011366,-0.034733
150,184,CHEMBL1255631,CC(=O)NCCCC(=O)N[C@@H](Cc1ccccc1)C(=O)N1Cc2ccc...,4.331614,Yersinia_pestis,-0.481857,-0.191430,0.262318,-0.137011,0.103338,...,0.040469,-0.042034,-0.031812,0.008757,-0.011936,0.032443,-0.033617,-0.005212,-0.042472,0.003975
151,186,CHEMBL1916867,CSC(=S)N/N=C/c1ccccc1O,5.397940,Yersinia_pestis,0.078611,0.289177,-0.020564,-0.485454,-0.047545,...,0.203329,-0.026297,0.016365,-0.033141,-0.087305,-0.039236,0.005247,-0.080502,-0.053136,-0.003147


### Prepare data to run `ART` on `Yersinia Pestis` data with `duplicate` SMILES

#### Find Features and Response

In [40]:
input_var = [col for col in YersiniaPestisData_chEMBL_wMACAW.columns if col.startswith('MACAW_')]
print(f"MACAW Embeddings: {len(input_var)}")
print(input_var)

MACAW Embeddings: 30
['MACAW_1', 'MACAW_2', 'MACAW_3', 'MACAW_4', 'MACAW_5', 'MACAW_6', 'MACAW_7', 'MACAW_8', 'MACAW_9', 'MACAW_10', 'MACAW_11', 'MACAW_12', 'MACAW_13', 'MACAW_14', 'MACAW_15', 'MACAW_16', 'MACAW_17', 'MACAW_18', 'MACAW_19', 'MACAW_20', 'MACAW_21', 'MACAW_22', 'MACAW_23', 'MACAW_24', 'MACAW_25', 'MACAW_26', 'MACAW_27', 'MACAW_28', 'MACAW_29', 'MACAW_30']


In [41]:
features = YersiniaPestisData_chEMBL_wMACAW[input_var].to_numpy()

In [42]:
response_var = ["pPotency"]
print(response_var)

['pPotency']


In [43]:
response = YersiniaPestisData_chEMBL_wMACAW[response_var].to_numpy()

### save the data as a EDD style file

In [44]:
utils.save_edd_csv(features, response, input_var, modelBuildingDataDir + 'YersiniaPestisData_chEMBL_wMACAW_ARTready.csv', response_var)

### Predict response with ART

In [45]:
YersiniaPestisData_chEMBL_wMACAW_ARTready = pd.read_csv(modelBuildingDataDir + "YersiniaPestisData_chEMBL_wMACAW_ARTready.csv")
YersiniaPestisData_chEMBL_wMACAW_ARTready

,Line Name,Type,0.0
0,0,MACAW_1,-0.481541
1,1,MACAW_1,-0.482748
2,2,MACAW_1,-0.482518
3,3,MACAW_1,-0.481617
4,4,MACAW_1,-0.003173
...,...,...,...
4738,148,pPotency,4.494850
4739,149,pPotency,4.494850
4740,150,pPotency,4.331614
4741,151,pPotency,5.397940


### Define the ART parameters needed for the prediction

In [46]:
art_params = {
    'input_vars': input_var,
    'response_vars': response_var,
    'objective': 'maximize',
    'threshold': 0.2,
    'alpha': 0.25,  # changed from 0.5 --> 0.25, 
    'num_recommendations': 10,
    'max_mcmc_cores': 4,
    'seed': 42,                    
    'output_dir': artResultDir,
    'recommend': False,
    'cross_val': True,
    'num_tpot_models': 2,
}

### Run ART without recommendations but with cross-validations to gauge how generalizable the results are

### Load pre-trained ART model (if necessary, otherwise skip this step)

In [50]:
ARTtrainedModelFile = os.path.join(artResultDir, 'art.cpkl')
with open(ARTtrainedModelFile, 'rb') as f:
    art = cloudpickle.load(f)
print(f" ART model loaded from: {ARTtrainedModelFile}")

 ART model loaded from: /users/sghosh6/DTRA_project/MACAW/DTRA_ART/test_CLI/ART_results/model_outputs/art.cpkl


## 2.5 Discovery of new hits specific to all Bacteriaes (data source `Retrotide` generated molecules)

In this section, we screen a custom virtual library looking for molecules that are promising accoring to the the SVR models `regr` above, which use 15-D MACAW embeddings as their input. The library of compounds has been generated using `Retrotide`. In particular, we are interested in molecules with high predicted pPotency. Here is the settings and corresponding notebook link:

0. Start from first PKS compound of Basidalin synthesis
1. Keep fixed module1
2. diversify only starter and second extension module
3. The TE domain is trying to cyclize the molecule, but the chemical reaction pattern doesn't match the molecule structure, so rxn.RunReactants() returns an empty result with 73.6% failure rate. So we are using different ring positionsto find most diverse set of molecules.

https://github.com/soumyadeepghosh35/RetroTide/blob/DTRA_Retrotide/notebooks/Basidalin_Retrotide.ipynb

Load MACAW embeedings for external validation data set

In [58]:
YersiniaPestis_RetrotideDatasets_wMACAW = pd.read_csv("/users/sghosh6/DTRA_project/MACAW/DTRA_ART/DrugDesignData/Results/Yersinia_pestis_test/YersiniaPestis_ChemDivAntiviralsData_wMACAW.csv")
YersiniaPestis_RetrotideDatasets_wMACAW = YersiniaPestis_RetrotideDatasets_wMACAW.iloc[:, :-15]
YersiniaPestis_RetrotideDatasets_wMACAW

,SMILES,MACAW_1,MACAW_2,MACAW_3,MACAW_4,MACAW_5,MACAW_6,MACAW_7,MACAW_8,MACAW_9,...,MACAW_21,MACAW_22,MACAW_23,MACAW_24,MACAW_25,MACAW_26,MACAW_27,MACAW_28,MACAW_29,MACAW_30
0,O=C(Oc1ccc(-c2ccccc2)cc1)c1ccc([N+](=O)[O-])cc1,0.059497,0.085250,-0.022410,-0.030475,-0.083338,-0.063058,0.086810,-0.050318,-0.027745,...,-0.001208,-0.034375,-0.023211,0.006280,-0.065071,-0.046923,0.019977,-0.086387,-0.024572,-0.009345
1,COc1ccc(/C=N/NC(=O)c2ccn(C)n2)cc1,0.075648,0.114266,-0.026053,-0.117645,-0.051425,0.033918,0.098375,-0.072583,-0.024356,...,-0.058210,0.100592,0.015594,0.046478,-0.077035,-0.002444,-0.041142,-0.022635,-0.045519,0.061474
2,CC1(C)OC1COc1cc(=O)oc2cc3occc3cc12,0.001241,0.133997,-0.053262,-0.049792,-0.104491,0.012692,0.075170,-0.122961,0.009274,...,-0.005927,0.043765,-0.020567,-0.007563,-0.068065,-0.032031,-0.033314,-0.040947,-0.032078,0.016981
3,O=C(Cc1ccccc1)c1ccc(-c2ccccc2)cc1,0.083241,0.094899,-0.000856,-0.056454,-0.063781,-0.085394,0.139605,-0.074898,-0.062061,...,0.004076,0.023714,-0.024719,-0.033143,-0.098792,-0.037075,-0.007793,-0.110907,0.017789,-0.031432
4,O=C(Nc1ccc(Oc2ccccc2)cc1)c1cccc(C(=O)Nc2ccc(Oc...,0.002239,0.028211,-0.013030,-0.008984,-0.068836,0.021507,0.125704,-0.078776,-0.047453,...,-0.026052,0.010365,-0.000717,-0.016849,-0.042909,-0.020137,0.024596,-0.112854,-0.059633,0.034594
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12505,Nc1nc(Nc2ccccc2F)nc2ccccc12,0.106329,0.105069,-0.009564,-0.126560,-0.058035,-0.121850,0.133765,-0.046374,-0.065447,...,-0.020598,0.064871,0.001215,-0.070307,-0.048691,0.070191,0.012920,-0.001458,0.037205,-0.102944
12506,CNc1nc(Nc2ccccc2F)nc2ccccc12,0.110776,0.100613,-0.004898,-0.125607,-0.055423,-0.054468,0.128465,-0.080077,-0.060959,...,0.007333,0.051002,0.025682,-0.033157,-0.036091,0.033382,0.012516,-0.000512,0.014743,-0.120702
12507,Nc1nc(Nc2ccccc2Cl)nc2ccccc12,0.095829,0.103267,-0.026141,-0.129554,-0.051299,-0.106580,0.170083,-0.064797,-0.061989,...,-0.037296,0.045535,0.040301,-0.046624,-0.065502,0.061704,0.001802,0.013268,0.081063,-0.093601
12508,CNc1nc(Nc2ccccc2Cl)nc2ccccc12,0.104935,0.095797,-0.019314,-0.123463,-0.045209,-0.038661,0.157588,-0.098184,-0.060461,...,0.008035,0.037263,0.055898,-0.018234,-0.041050,0.006605,-0.006695,0.012771,0.045838,-0.107181


In [59]:
macaw_columns = [col for col in YersiniaPestis_RetrotideDatasets_wMACAW.columns if col.startswith('MACAW_')]
smi_lib_RetrotideDatasets_wMACAW = YersiniaPestis_RetrotideDatasets_wMACAW[macaw_columns].values

Get predictions with uncertainty using ART's post_pred_stats

In [60]:
# Pass X1_lib (features)
mean, std = art.post_pred_stats(smi_lib_RetrotideDatasets_wMACAW)

# Now use these for your results
RetrotideAntiBioticData_predicted = YersiniaPestis_RetrotideDatasets_wMACAW[['SMILES']].copy()
RetrotideAntiBioticData_predicted['pPotency_prediction'] = mean
RetrotideAntiBioticData_predicted['pPotency_std'] = std

# Calculate 95% Confidence Intervals
RetrotideAntiBioticData_predicted['pPotency_lower_95CI'] = mean - 1.96 * std
RetrotideAntiBioticData_predicted['pPotency_upper_95CI'] = mean + 1.96 * std

# Convert to IC50
RetrotideAntiBioticData_predicted['IC50(M)_prediction'] = 10 ** (-mean)
RetrotideAntiBioticData_predicted['IC50(M)_lower_95CI'] = 10 ** (-RetrotideAntiBioticData_predicted['pPotency_upper_95CI'])
RetrotideAntiBioticData_predicted['IC50(M)_upper_95CI'] = 10 ** (-RetrotideAntiBioticData_predicted['pPotency_lower_95CI'])

# Select and save results
RetrotideAntiBioticData_predicted = RetrotideAntiBioticData_predicted.filter(
    items=["SMILES", "pPotency_prediction", "pPotency_std", 
           "pPotency_lower_95CI", "pPotency_upper_95CI",
           "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI"]
)

RetrotideAntiBioticData_predicted.to_csv(os.path.join(resultsDir + "YersiniaPestis_wART_RetrotideDataset_predicted_all.csv"), index=False)
RetrotideAntiBioticData_predicted[['SMILES']].to_csv(os.path.join(resultsDir + "YersiniaPestis_wART_RetrotideDataset_predicted_all_SMILES.csv"), index=False)
print(f"Predictions saved with uncertainty estimates")
RetrotideAntiBioticData_predicted

Predictions saved with uncertainty estimates


,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI
0,O=C(Oc1ccc(-c2ccccc2)cc1)c1ccc([N+](=O)[O-])cc1,4.260295,1.335220,1.643265,6.877326,0.000055,1.326400e-07,0.022737
1,COc1ccc(/C=N/NC(=O)c2ccn(C)n2)cc1,4.080000,1.336055,1.461331,6.698668,0.000083,2.001391e-07,0.034568
2,CC1(C)OC1COc1cc(=O)oc2cc3occc3cc12,4.043454,1.335719,1.425444,6.661465,0.000090,2.180396e-07,0.037545
3,O=C(Cc1ccccc1)c1ccc(-c2ccccc2)cc1,4.273331,1.342287,1.642447,6.904214,0.000053,1.246769e-07,0.022780
4,O=C(Nc1ccc(Oc2ccccc2)cc1)c1cccc(C(=O)Nc2ccc(Oc...,4.285548,1.332611,1.673631,6.897466,0.000052,1.266292e-07,0.021202
...,...,...,...,...,...,...,...,...
12505,Nc1nc(Nc2ccccc2F)nc2ccccc12,4.329293,1.339042,1.704771,6.953814,0.000047,1.112207e-07,0.019735
12506,CNc1nc(Nc2ccccc2F)nc2ccccc12,4.365802,1.338491,1.742359,6.989244,0.000043,1.025075e-07,0.018098
12507,Nc1nc(Nc2ccccc2Cl)nc2ccccc12,4.280111,1.339527,1.654637,6.905584,0.000052,1.242841e-07,0.022149
12508,CNc1nc(Nc2ccccc2Cl)nc2ccccc12,4.299892,1.337764,1.677875,6.921909,0.000050,1.196992e-07,0.020995
